In [6]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "ma_walk_forward_equity"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

pd.set_option(
    "display.max_columns",
    100,
)

pd.set_option(
    "display.width",
    180,
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:.4f}",
)

plt.rcParams[
    "axes.unicode_minus"
] = False

import matplotlib.pyplot as plt
from matplotlib import font_manager


def configure_chinese_font() -> str | None:
    """
    从当前系统已安装字体中选择一个支持中文的字体。
    """
    preferred_fonts = [
        "Microsoft YaHei",
        "Microsoft YaHei UI",
        "SimHei",
        "SimSun",
        "Noto Sans CJK SC",
        "Source Han Sans SC",
        "Arial Unicode MS",
    ]

    installed_fonts = {
        font.name
        for font in font_manager.fontManager.ttflist
    }

    for font_name in preferred_fonts:
        if font_name in installed_fonts:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [
                font_name
            ]
            plt.rcParams["axes.unicode_minus"] = False

            print(
                "Matplotlib 中文字体：",
                font_name,
            )

            return font_name

    print(
        "没有发现可用的中文字体，"
        "图中的中文可能无法正常显示"
    )

    return None


chinese_font = configure_chinese_font()


%load_ext autoreload
%autoreload 2

from walk_forward_equity import (
    build_dynamic_schedule,
    build_fixed_schedule,
    build_continuous_stock_detail,
    build_portfolio,
    summarize_portfolio,
    summarize_annual_returns,
    build_equity_comparison,
    plot_equity_curves,
    plot_drawdown_curves,
    summarize_parameter_validation
)

from parameter_sensitivity import (
    build_ma_parameter_grid,
)

from walk_forward import (
    generate_walk_forward_windows,
    run_ma_walk_forward,
    run_fixed_parameter_validation,
    plot_walk_forward_metric,
    plot_selected_parameters
)

from backtest import (ma_cross_backtest,
                      backtest_plot_drawdown,
                      backtest_plot_nav,
                      backtest_plot_sbpoints,
                      calculate_performance,
                      calculate_buy_and_sell,
                      format_batch_summary,
                      run_batch_ma_backtest,
                      summarize_backtest_period,
                      run_ma_parameter_grid_search)

Matplotlib 中文字体： Microsoft YaHei
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
validation_stock_list = [
    "600028",  # 中国石化：石油石化
    "600276",  # 恒瑞医药：医药
    "600104",  # 上汽集团：汽车
    "600050",  # 中国联通：通信
    "601006",  # 大秦铁路：交通运输
    "600887",  # 伊利股份：食品乳业
    "600031",  # 三一重工：工程机械
    "600585",  # 海螺水泥：建筑材料
    "600900",  # 长江电力：公用事业
    "600570",  # 恒生电子：软件
]

validation_parameters = [
    (10, 40),  # 当前主要候选
    (20, 60),  # 原始基准
]

In [3]:
parameter_runs = {}

for fast_window, slow_window in validation_parameters:
    print(
        f"\n开始回测参数："
        f"{fast_window}/{slow_window}"
    )

    batch_summary, batch_results = (
        run_batch_ma_backtest(
            stock_list=validation_stock_list,
            fast_window=fast_window,
            slow_window=slow_window,
            commission_rate=0.0003,
            slippage_rate=0.0002,
            annual_risk_free_rate=0.0,
            trading_days=252,
            save_result=False,
        )
    )

    parameter_runs[
        (fast_window, slow_window)
    ] = {
        "summary": batch_summary,
        "results": batch_results,
    }


开始回测参数：10/40
正在回测：600028
正在回测：600276
正在回测：600104
正在回测：600050
正在回测：601006
正在回测：600887
正在回测：600031
正在回测：600585
正在回测：600900
正在回测：600570

开始回测参数：20/60
正在回测：600028
正在回测：600276
正在回测：600104
正在回测：600050
正在回测：601006
正在回测：600887
正在回测：600031
正在回测：600585
正在回测：600900
正在回测：600570


In [4]:

parameter_summary_frames = []

for (
    fast_window,
    slow_window,
), run_result in parameter_runs.items():

    parameter_summary = (
        run_result["summary"].copy()
    )

    parameter_summary[
        "fast_window"
    ] = fast_window

    parameter_summary[
        "slow_window"
    ] = slow_window

    parameter_summary[
        "ma_param"
    ] = (
        f"{fast_window}/"
        f"{slow_window}"
    )

    parameter_summary_frames.append(
        parameter_summary
    )

validation_detail = pd.concat(
    parameter_summary_frames,
    ignore_index=True,
)

In [5]:
display(
    validation_detail[
        [
            "symbol",
            "ma_param",
            "strategy_annual_return",
            "strategy_sharpe",
            "strategy_max_drawdown",
            "excess_annual_return",
            "sharpe_diff",
            "drawdown_improvement",
            "exposure",
            "total_trade_count",
            "total_transaction_cost",
        ]
    ].sort_values(
        [
            "symbol",
            "ma_param",
        ]
    )
)

,symbol,ma_param,strategy_annual_return,strategy_sharpe,strategy_max_drawdown,excess_annual_return,sharpe_diff,drawdown_improvement,exposure,total_trade_count,total_transaction_cost
4,600028,10/40,0.0213,0.2054,-0.3690,-0.0223,-0.0903,0.1209,0.5310,82,0.0410
14,600028,20/60,0.0108,0.1525,-0.5571,-0.0255,-0.1155,-0.0673,0.5706,56,0.0280
2,600031,10/40,0.0700,0.3813,-0.6046,-0.0236,-0.0434,0.1146,0.5515,82,0.0410
12,600031,20/60,0.0822,0.4191,-0.5692,0.0072,0.0389,0.1501,0.5606,50,0.0250
7,600050,10/40,-0.0262,0.0395,-0.6333,-0.0255,-0.1366,0.0237,0.4249,86,0.0430
18,600050,20/60,-0.0409,-0.0309,-0.6421,-0.0274,-0.1708,0.0148,0.4036,58,0.0290
9,600104,10/40,-0.0575,-0.1541,-0.6699,-0.0241,-0.2050,-0.0160,0.4728,86,0.0430
15,600104,20/60,-0.0060,0.0851,-0.4210,0.0405,0.0770,0.2330,0.4719,54,0.0270
0,600276,10/40,0.1732,0.7550,-0.3305,0.0001,0.1217,0.3794,0.5740,75,0.0375
11,600276,20/60,0.1306,0.6088,-0.3555,-0.0287,0.0097,0.3544,0.6077,48,0.0240


In [7]:
validation_summary = (
    summarize_parameter_validation(
        validation_detail
    )
)

display(validation_summary)

,ma_param,stock_count,avg_strategy_annual_return,median_strategy_annual_return,std_strategy_annual_return,avg_strategy_sharpe,median_strategy_sharpe,std_strategy_sharpe,worst_strategy_sharpe,avg_excess_annual_return,median_excess_annual_return,std_excess_annual_return,worst_excess_annual_return,avg_sharpe_diff,median_sharpe_diff,avg_strategy_max_drawdown,avg_drawdown_improvement,worst_drawdown_improvement,avg_exposure,avg_trade_count,avg_transaction_cost,return_win_rate,sharpe_win_rate,drawdown_win_rate
0,10/40,10,0.0335,0.0202,0.0686,0.2532,0.2009,0.3019,-0.1541,-0.0188,-0.0238,0.0366,-0.0755,-0.0867,-0.1187,-0.4822,0.1164,-0.0160,0.5165,83.1000,0.0416,0.2000,0.2000,0.9000
1,20/60,10,0.0230,0.0024,0.0621,0.2193,0.1188,0.2612,-0.0689,-0.0137,-0.0239,0.0319,-0.0572,-0.0763,-0.0497,-0.4996,0.0990,-0.0673,0.5299,52.1000,0.0261,0.3000,0.3000,0.8000


In [8]:
comparison_10_40 = validation_detail.loc[
    validation_detail["ma_param"]
    == "10/40"
].copy()

comparison_20_60 = validation_detail.loc[
    validation_detail["ma_param"]
    == "20/60"
].copy()

paired_comparison = (
    comparison_10_40.merge(
        comparison_20_60,
        on="symbol",
        how="inner",
        suffixes=(
            "_10_40",
            "_20_60",
        ),
    )
)

paired_comparison[
    "annual_return_diff"
] = (
    paired_comparison[
        "strategy_annual_return_10_40"
    ]
    - paired_comparison[
        "strategy_annual_return_20_60"
    ]
)

paired_comparison[
    "sharpe_diff_between_params"
] = (
    paired_comparison[
        "strategy_sharpe_10_40"
    ]
    - paired_comparison[
        "strategy_sharpe_20_60"
    ]
)

paired_comparison[
    "excess_return_diff"
] = (
    paired_comparison[
        "excess_annual_return_10_40"
    ]
    - paired_comparison[
        "excess_annual_return_20_60"
    ]
)

paired_comparison[
    "drawdown_diff_between_params"
] = (
    paired_comparison[
        "strategy_max_drawdown_10_40"
    ]
    - paired_comparison[
        "strategy_max_drawdown_20_60"
    ]
)

paired_comparison[
    "trade_count_diff"
] = (
    paired_comparison[
        "total_trade_count_10_40"
    ]
    - paired_comparison[
        "total_trade_count_20_60"
    ]
)

display(
    paired_comparison[
        [
            "symbol",
            "strategy_annual_return_10_40",
            "strategy_annual_return_20_60",
            "annual_return_diff",
            "strategy_sharpe_10_40",
            "strategy_sharpe_20_60",
            "sharpe_diff_between_params",
            "excess_annual_return_10_40",
            "excess_annual_return_20_60",
            "excess_return_diff",
            "strategy_max_drawdown_10_40",
            "strategy_max_drawdown_20_60",
            "drawdown_diff_between_params",
            "trade_count_diff",
        ]
    ]
)

,symbol,strategy_annual_return_10_40,strategy_annual_return_20_60,annual_return_diff,strategy_sharpe_10_40,strategy_sharpe_20_60,sharpe_diff_between_params,excess_annual_return_10_40,excess_annual_return_20_60,excess_return_diff,strategy_max_drawdown_10_40,strategy_max_drawdown_20_60,drawdown_diff_between_params,trade_count_diff
0,600276,0.1732,0.1306,0.0426,0.7550,0.6088,0.1462,0.0001,-0.0287,0.0288,-0.3305,-0.3555,0.0250,27
1,600900,0.1038,0.0877,0.0161,0.7222,0.6102,0.1120,-0.0447,-0.0510,0.0063,-0.1497,-0.2270,0.0773,27
2,600031,0.0700,0.0822,-0.0123,0.3813,0.4191,-0.0379,-0.0236,0.0072,-0.0307,-0.6046,-0.5692,-0.0355,32
3,600570,0.0505,-0.0338,0.0843,0.3146,0.0814,0.2332,0.0680,0.0277,0.0403,-0.6193,-0.6952,0.0758,40
4,600028,0.0213,0.0108,0.0104,0.2054,0.1525,0.0529,-0.0223,-0.0255,0.0032,-0.3690,-0.5571,0.1881,26
5,600887,0.0191,0.0588,-0.0397,0.1963,0.3643,-0.1680,-0.0755,-0.0224,-0.0532,-0.5159,-0.4808,-0.0351,36
6,600585,0.0018,-0.0406,0.0425,0.1278,-0.0689,0.1968,-0.0294,-0.0572,0.0278,-0.5315,-0.6224,0.0909,38
7,600050,-0.0262,-0.0409,0.0147,0.0395,-0.0309,0.0704,-0.0255,-0.0274,0.0019,-0.6333,-0.6421,0.0088,28
8,601006,-0.0215,-0.0191,-0.0024,-0.0559,-0.0285,-0.0274,-0.0112,-0.0004,-0.0108,-0.3984,-0.4260,0.0276,24
9,600104,-0.0575,-0.0060,-0.0516,-0.1541,0.0851,-0.2393,-0.0241,0.0405,-0.0646,-0.6699,-0.4210,-0.2489,32


In [9]:
paired_result_summary = pd.DataFrame(
    {
        "comparison": [
            "10/40 年化收益更高",
            "10/40 夏普更高",
            "10/40 超额收益更高",
            "10/40 最大回撤更好",
            "10/40 交易次数更少",
        ],
        "stock_count": [
            (
                paired_comparison[
                    "annual_return_diff"
                ] > 0
            ).sum(),

            (
                paired_comparison[
                    "sharpe_diff_between_params"
                ] > 0
            ).sum(),

            (
                paired_comparison[
                    "excess_return_diff"
                ] > 0
            ).sum(),

            (
                paired_comparison[
                    "drawdown_diff_between_params"
                ] > 0
            ).sum(),

            (
                paired_comparison[
                    "trade_count_diff"
                ] < 0
            ).sum(),
        ],
    }
)

paired_result_summary[
    "total_stock_count"
] = len(paired_comparison)

paired_result_summary[
    "win_rate"
] = (
    paired_result_summary[
        "stock_count"
    ]
    / paired_result_summary[
        "total_stock_count"
    ]
)

display(paired_result_summary)

,comparison,stock_count,total_stock_count,win_rate
0,10/40 年化收益更高,6,10,0.6000
1,10/40 夏普更高,6,10,0.6000
2,10/40 超额收益更高,6,10,0.6000
3,10/40 最大回撤更好,7,10,0.7000
4,10/40 交易次数更少,0,10,0.0000


## 新股票池外部验证结论

固定参数 `10/40` 和 `20/60` 在 10 只新股票上的超额收益胜率分别为 20% 和 30%，整体表现均较差。

`10/40` 在部分收益、夏普或回撤指标上略优于 `20/60`，但两组参数的总体差异不大，且均未表现出稳定的跨股票超额收益能力。

因此，`10/40` 在原始股票池中的优势未能迁移到新的行业和股票样本。当前结果表明，之前发现的最优参数具有较强的样本依赖性，不能作为适用于不同股票的通用均线参数。

本轮外部股票池验证判定为未通过。后续不再使用这 10 只股票重新搜索均线参数，以避免将验证样本重新变为训练样本。